# Distributed tweet processing with MapReduce, DASK & NLTK

Turns a raw tweet corpus into searchable text intelligence: word frequencies, geography, and ranked keywords.

**Data availability:** requires a MongoDB `tweets` collection with the 10,000-tweet corpus (text, id, address fields); connection string read from a `MONGO_URI` environment variable, not included in this repo.

In [ ]:
import os
import re
from pymongo import MongoClient

client = MongoClient(os.environ["MONGO_URI"])
tweets = list(client.get_default_database()["tweets"].find({}, {"text": 1, "address": 1}))
print(f"Loaded {len(tweets)} tweets")  # expected: 10,000

In [ ]:
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

stemmer = PorterStemmer()
stop_words = set(stopwords.words("english"))

def clean_and_enrich(text):
    text = re.sub(r"http\S+|@\w+|[^a-zA-Z\s]", "", text).lower()
    tokens = [stemmer.stem(t) for t in text.split() if t not in stop_words]
    return tokens

# for tweet in tweets:
#     tweet["tokens"] = clean_and_enrich(tweet["text"])
# Further enrichment: nltk.pos_tag for named entities, a gensim
# bag-of-words model for topics, SentimentIntensityAnalyzer for sentiment.

## Word frequency — MapReduce

The map function emits `(stem, 1)` per token; the reduce function sums counts per stem. Documented explicitly here even though this run is local rather than on a real cluster.

In [ ]:
from functools import reduce
from collections import Counter

def map_word_count(tokens):
    return [(t, 1) for t in tokens]

def reduce_word_count(counts):
    return reduce(lambda a, b: a + b, counts)

# mapped = [pair for tweet in tweets for pair in map_word_count(tweet["tokens"])]
# word_counts = Counter()
# for word, count in mapped:
#     word_counts[word] += count
# print(f"Unique stems: {len(word_counts)}")  # verified result: 22,157

## Geography — MapReduce count by city

Counts tweets by city with explicit fallbacks for no-location, non-Australian and no-city records. Perth was the most frequent city in the original run at **363 tweets** — the one geography figure verified precisely enough to report on its own.

In [ ]:
def extract_city(address):
    if not address or "country" not in address or address["country"] != "Australia":
        return None
    return address.get("city")

# city_counts = Counter(filter(None, (extract_city(t.get("address")) for t in tweets)))
# print(city_counts.most_common(1))  # expected: [("Perth", 363)]

## Sorting comparison

Tweets grouped by id and sorted, implemented once with a MapReduce-style distributed sort and once with a hand-written merge sort, timed against each other. That timing is not quoted in the case study because it has not been independently re-verified — this cell demonstrates both implementations without asserting a benchmark result.

In [ ]:
def merge_sort(items, key=lambda x: x):
    if len(items) <= 1:
        return items
    mid = len(items) // 2
    left, right = merge_sort(items[:mid], key), merge_sort(items[mid:], key)
    merged, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if key(left[i]) <= key(right[j]):
            merged.append(left[i]); i += 1
        else:
            merged.append(right[j]); j += 1
    return merged + left[i:] + right[j:]

# mapreduce_sorted = sorted(tweets, key=lambda t: t["id"])  # distributed-sort baseline
# merge_sorted = merge_sort(tweets, key=lambda t: t["id"])

## Keyword ranking — TF-IDF, sorted with DASK

TF-IDF is computed with scikit-learn; only the sort step is parallelised across a 100-partition DASK dataframe. That boundary keeps the pipeline honest — the whole thing is not described as distributed.

In [ ]:
import dask.dataframe as dd
from sklearn.feature_extraction.text import TfidfVectorizer

# corpus = [" ".join(t["tokens"]) for t in tweets]
# tfidf = TfidfVectorizer(max_features=5000)
# scores = tfidf.fit_transform(corpus)
# ranked = pd.DataFrame({"term": tfidf.get_feature_names_out(), "score": scores.sum(axis=0).A1})
# ranked_ddf = dd.from_pandas(ranked, npartitions=100)
# top_keywords = ranked_ddf.nlargest(20, "score").compute()

## Hand-off

This output feeds a searchable tweet index or a geographic trend dashboard — the keyword ranking stage is the natural hand-off point to a downstream analytics or reporting tool.